# run test

In [7]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings

# Suppress minor formatting warnings for cleaner output
warnings.filterwarnings('ignore') 

# 1. Map your 5 models to their respective CSV files
files = {
    'SD-Net': 'sdnet_evaluation_results.csv',
    'YOLOv7-tiny': 'yolov7_tiny_evaluation_results.csv',
    'YOLOv9': 'yolov9_evaluation_results.csv',
    'CenterNet': 'centernet_evaluation_results.csv',
    'Faster R-CNN': 'fasterrcnn_evaluation_results.csv'
}

dataframes = []

# Load and label the data
for model_name, file_path in files.items():
    try:
        df          = pd.read_csv(file_path)
        df['Model'] = model_name
        dataframes.append(df)
    except FileNotFoundError:
        print(f"Error: Could not find '{file_path}'. Please ensure it is in the same directory.")

# Combine everything into one unified DataFrame
df_combined = pd.concat(dataframes, ignore_index=True)
# print(df_combined.head())  # Optional: Check the combined data structure    

# 2. Filter strictly for the overall ('All') class performance
# We use str.lower() to catch both 'all' and 'All' across different frameworks
df_anova = df_combined[df_combined['Class'].astype(str).str.lower() == 'all'].copy()

# Ensure the mAP columns are numeric
df_anova['mAP50']    = pd.to_numeric(df_anova['mAP50'])
df_anova['mAP50-95'] = pd.to_numeric(df_anova['mAP50-95'])

# 3. Define the testing function
def run_tests(metric):
    print(f"\n{'='*80}")
    print(f"STATISTICAL ANALYSIS FOR: {metric}")
    print(f"{'='*80}")

    # Remove any missing data just in case
    clean_df = df_anova.dropna(subset=[metric])

    # Extract arrays of data for each model to feed into the ANOVA
    groups = [group[metric].values for name, group in clean_df.groupby('Model')]
    # print(f"Data groups for ANOVA (by model): {[group.shape[0] for group in groups]} samples each")

    # -- ONE-WAY ANOVA --
    f_stat, p_val = stats.f_oneway(*groups)
    print(f"One-Way ANOVA F-statistic: {f_stat:.4f}")
    print(f"One-Way ANOVA p-value:     {p_val:.4e}")

    # If the ANOVA p-value is significant (typically < 0.05), run post-hoc
    if p_val < 0.05:
        print("\n[RESULT] The ANOVA p-value is < 0.05. Significant differences exist.")
        print("Executing Tukey's HSD Post-Hoc Test to identify specific model differences...\n")

        # -- TUKEY'S HSD --
        tukey = pairwise_tukeyhsd(endog=clean_df[metric],
                                  groups=clean_df['Model'],
                                  alpha=0.05)
        print(tukey)
    else:
        print("\n[RESULT] The ANOVA p-value is >= 0.05. No statistically significant differences were found.")

# 4. Execute tests for both evaluation metrics
run_tests('mAP50')
run_tests('mAP50-95')


STATISTICAL ANALYSIS FOR: mAP50
One-Way ANOVA F-statistic: 19.2675
One-Way ANOVA p-value:     1.2324e-06

[RESULT] The ANOVA p-value is < 0.05. Significant differences exist.
Executing Tukey's HSD Post-Hoc Test to identify specific model differences...

      Multiple Comparison of Means - Tukey HSD, FWER=0.05      
   group1       group2    meandiff p-adj   lower  upper  reject
---------------------------------------------------------------
   CenterNet Faster R-CNN    0.004 0.5649 -0.0039 0.0119  False
   CenterNet       SD-Net   0.0022  0.917 -0.0057 0.0101  False
   CenterNet  YOLOv7-tiny   0.0168    0.0  0.0089 0.0247   True
   CenterNet       YOLOv9   0.0168    0.0  0.0089 0.0247   True
Faster R-CNN       SD-Net  -0.0018 0.9581 -0.0097 0.0061  False
Faster R-CNN  YOLOv7-tiny   0.0128 0.0008  0.0049 0.0207   True
Faster R-CNN       YOLOv9   0.0128 0.0008  0.0049 0.0207   True
      SD-Net  YOLOv7-tiny   0.0146 0.0002  0.0067 0.0225   True
      SD-Net       YOLOv9   0.0146 0.0002

In [2]:
# --- Cell 4: Calculate Mean ± Std for the manuscript ---

# Group by the 'Model' column and calculate the mean and std for both metrics
summary_stats = df_anova.groupby('Model')[['mAP50', 'mAP50-95']].agg(['mean', 'std'])

print("="*65)
print(f"{'OVERALL PERFORMANCE: Mean ± Std (5 Seeds)':^65}")
print("="*65)
print(f"{'Model':<15} | {'mAP@0.5 (%)':<20} | {'mAP@0.5:0.95 (%)':<20}")
print("-" * 65)

# Iterate through each model and format the output
for model in summary_stats.index:
    # Extract values and convert to percentages
    m50_mean = summary_stats.loc[model, ('mAP50', 'mean')] * 100
    m50_std  = summary_stats.loc[model, ('mAP50', 'std')] * 100
    
    m95_mean = summary_stats.loc[model, ('mAP50-95', 'mean')] * 100
    m95_std  = summary_stats.loc[model, ('mAP50-95', 'std')] * 100
    
    # Format the strings to 2 decimal places (e.g., "96.12 ± 0.30")
    res_50 = f"{m50_mean:.2f} ± {m50_std:.2f}"
    res_95 = f"{m95_mean:.2f} ± {m95_std:.2f}"
    
    print(f"{model:<15} | {res_50:<20} | {res_95:<20}")

print("="*65)

            OVERALL PERFORMANCE: Mean ± Std (5 Seeds)            
Model           | mAP@0.5 (%)          | mAP@0.5:0.95 (%)    
-----------------------------------------------------------------
CenterNet       | 95.90 ± 0.63         | 76.82 ± 1.36        
Faster R-CNN    | 96.30 ± 0.39         | 75.06 ± 0.98        
SD-Net          | 96.12 ± 0.30         | 75.84 ± 0.47        
YOLOv7-tiny     | 97.58 ± 0.29         | 80.72 ± 1.11        
YOLOv9          | 97.58 ± 0.38         | 86.10 ± 0.41        
